In [ ]:

!pip install pypdf


In [ ]:

from google.colab import files

uploaded = files.upload()


Saving embeddings.pdf to embeddings (1).pdf


In [ ]:

from pypdf import PdfReader

# Get the name of the uploaded PDF
pdf_name = next(iter(uploaded))

# Open the PDF
reader = PdfReader(pdf_name)

# Extract text from all pages
pdf_text = ""

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    pdf_text += text + "\n"

# Display the results
print("PDF Name:", pdf_name)
print("Total Pages:", len(reader.pages))
print("\nExtracted Text:\n")
print(pdf_text[:3000])


PDF Name: embeddings (1).pdf
Total Pages: 3

Extracted Text:

 
embeddings, and search for relevant passages using Python in Google Colab. 
1. What is Artificial Intelligence? 
Artificial Intelligence (AI) is a field of computing focused on building systems that perform 
tasks commonly 
associated with human intelligence. Examples include understanding language, 
recognizing images, making 
predictions, and helping people find information. An AI system is only as reliable as its 
design, data, and 
evaluation. 
2. What is Machine Learning? 
Machine Learning (ML) is a branch of AI in which computers learn patterns from data 
instead of relying only on 
hand-written rules. A model is trained using examples and then evaluated on data it has 
not seen before. 
Common ML tasks include classification, regression, clustering, and recommendation. 
3. Supervised Learning 
Supervised learning uses examples that contain inputs and known target answers. For 
example, a model can 
learn from house 

In [ ]:

def ask_pdf(question):
    # Convert the PDF text and question to lowercase
    document = pdf_text.lower()
    question = question.lower()

    # Split the question into individual words
    words = question.split()

    # Find PDF sentences that contain question words
    sentences = document.replace("?", ".").replace("!", ".").split(".")

    matching_sentences = []

    for sentence in sentences:
        sentence = sentence.strip()

        if any(word in sentence for word in words if len(word) > 3):
            matching_sentences.append(sentence)

    # Display the results
    if matching_sentences:
        print("Possible answer from your PDF:\n")

        for sentence in matching_sentences[:5]:
            print("-", sentence)
    else:
        print("I couldn't find a matching sentence in your PDF.")


In [ ]:

ask_pdf("What is this document about?")


Possible answer from your PDF:

- what is artificial intelligence
- what is machine learning
- semantic search embeds a user's 
question and 
compares it with embedded document passages
- check 
whether retrieved 
passages are relevant, whether generated answers are supported by the document, and 
whether the system 
admits uncertainty when evidence is missing
- test different question wordings, short and 
long documents, and 
pdfs with empty or unreadable pages


In [ ]:
ask_pdf("What is machine learning")

Possible answer from your PDF:

- what is artificial intelligence
- what is machine learning
- machine learning (ml) is a branch of ai in which computers learn patterns from data 
instead of relying only on 
hand-written rules
- supervised learning 
supervised learning uses examples that contain inputs and known target answers
- unsupervised learning 
unsupervised learning looks for patterns in data without using labelled target answers


In [ ]:

!pip install -q sentence-transformers


In [ ]:

# Divide the PDF into chunks of 500 characters
chunk_size = 500
overlap = 100

chunks = []

for start in range(0, len(pdf_text), chunk_size - overlap):
    chunk = pdf_text[start:start + chunk_size]

    if chunk.strip():
        chunks.append(chunk)

print("Total chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0] if chunks else "No text found")


Total chunks: 11

First chunk:

 
embeddings, and search for relevant passages using Python in Google Colab. 
1. What is Artificial Intelligence? 
Artificial Intelligence (AI) is a field of computing focused on building systems that perform 
tasks commonly 
associated with human intelligence. Examples include understanding language, 
recognizing images, making 
predictions, and helping people find information. An AI system is only as reliable as its 
design, data, and 
evaluation. 
2. What is Machine Learning? 
Machine Learnin


In [ ]:

from sentence_transformers import SentenceTransformer

# Load a pretrained embedding model
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Convert each PDF chunk into an embedding
chunk_embeddings = model.encode(
    chunks,
    convert_to_tensor=True
)

print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 11
Embedding shape: torch.Size([11, 384])


In [ ]:

from sentence_transformers import util

def search_pdf(question, top_k=3):
    if not chunks:
        print("No PDF text is available.")
        return

    # Convert the question into an embedding
    question_embedding = model.encode(
        question,
        convert_to_tensor=True
    )

    # Compare the question with PDF chunks
    scores = util.cos_sim(
        question_embedding,
        chunk_embeddings
    )[0]

    # Get the best matching chunks
    k = min(top_k, len(chunks))
    best_matches = scores.topk(k)

    print("Relevant information from your PDF:\n")

    for score, index in zip(
        best_matches.values,
        best_matches.indices
    ):
        print(f"Similarity score: {score.item():.3f}")
        print(chunks[index.item()])
        print("-" * 50)


In [ ]:

search_pdf("What is the main topic of this document?")


Relevant information from your PDF:

Similarity score: 0.318
ing 
application. 
End of practice document. Use this PDF to test extraction, chunking, semantic search, and 
later LLM-based 
 

--------------------------------------------------
Similarity score: 0.297
ocuments to services unless you are permitted to do so. 
10. Quick Practice Questions 
1. What is the difference between AI and Machine Learning? 2. How does supervised 
learning use labelled 
examples? 3. What is an embedding? 4. What are the two main stages in a basic RAG 
system? 5. Why does a 
high similarity score not guarantee a correct answer? 6. Name two ways to test a PDF 
question-answering 
application. 
End of practice document. Use this PDF to test extraction, chunking, semantic sea
--------------------------------------------------
Similarity score: 0.233
formation retrieval 
with a language 
model. First, the system searches a knowledge source for relevant passages. Next, it 
provides those passages 
and the u

In [ ]:
search_pdf("What is the difference between AI and Machine Learning?")

Relevant information from your PDF:

Similarity score: 0.665
ly as reliable as its 
design, data, and 
evaluation. 
2. What is Machine Learning? 
Machine Learning (ML) is a branch of AI in which computers learn patterns from data 
instead of relying only on 
hand-written rules. A model is trained using examples and then evaluated on data it has 
not seen before. 
Common ML tasks include classification, regression, clustering, and recommendation. 
3. Supervised Learning 
Supervised learning uses examples that contain inputs and known target answers. For 
e
--------------------------------------------------
Similarity score: 0.640
 
embeddings, and search for relevant passages using Python in Google Colab. 
1. What is Artificial Intelligence? 
Artificial Intelligence (AI) is a field of computing focused on building systems that perform 
tasks commonly 
associated with human intelligence. Examples include understanding language, 
recognizing images, making 
predictions, and helping people

In [ ]:
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)

print("Gemini connected!")

Enter your Gemini API key: ··········
Gemini connected!


In [ ]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

In [ ]:

response = client.models.generate_content(
    model="gemini-flash-lite-latest",
    contents="Explain Artificial Intelligence in simple words."
)

print(response.text)


Imagine trying to teach a toddler how to recognize a cat. You show them pictures of cats, point out their whiskers, and say "cat." After seeing a few, the toddler starts recognizing cats on their own—even if it's a fluffy white one they've never seen before. 

**Artificial Intelligence (AI) is basically doing the same thing, but with computers.**

At its core, AI is about making computers "smart" so they can solve problems, learn, and make decisions much like humans do. 

Here are the 3 main things to know about AI:

### 1. It learns from experience
Instead of a human programmer writing strict, complicated rules for a computer to follow, you feed the computer massive amounts of data. The computer looks for patterns in that data. The more data it sees, the better it gets at its job. 

### 2. You use it every day (without realizing it)
AI isn't just about killer robots in sci-fi movies. It's already all around us:
* **Netflix & YouTube:** The recommendations for what you should watch nex

In [ ]:

from sentence_transformers import SentenceTransformer

# Give the embedding model its own variable name
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [ ]:

# Create embeddings for the PDF chunks
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_tensor=True
)

print("PDF embeddings created successfully!")
print("Number of chunks:", len(chunks))


PDF embeddings created successfully!
Number of chunks: 11


In [ ]:

from sentence_transformers import util

def ask_ai_pdf(question, top_k=3):
    if not chunks:
        print("Please upload a PDF with readable text.")
        return

    # 1. Convert the question into an embedding
    question_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True
    )

    # 2. Find the most relevant PDF sections
    scores = util.cos_sim(
        question_embedding,
        chunk_embeddings
    )[0]

    k = min(top_k, len(chunks))
    best_matches = scores.topk(k)

    context = "\n\n".join(
        chunks[index.item()]
        for index in best_matches.indices
    )

    # 3. Ask Gemini to answer using the retrieved text
    prompt = f"""
You are an AI PDF assistant.

Answer using only the PDF text below.
If the answer is not supported by the text,
say that the information was not found.
Do not invent facts.

PDF text:
{context}

Question:
{question}
"""

    response = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt
    )

    print("Question:", question)
    print("\nAI Answer:\n")
    print(response.text)


# Test your assistant
ask_ai_pdf("What is Artificial Intelligence?")


Question: What is Artificial Intelligence?

AI Answer:

Based on the provided text, Artificial Intelligence (AI) is a field of computing focused on building systems that perform tasks commonly associated with human intelligence. Examples include understanding language, recognizing images, making predictions, and helping people find information.


In [ ]:

!pip install -q ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.7 MB/s eta 0:00:00


In [53]:

import html
import ipywidgets as widgets
from IPython.display import display

# ---------- Header ----------
header = widgets.HTML("""
<div style="background:linear-gradient(135deg,#172554,#2563eb);
padding:24px;border-radius:14px;color:white;font-family:Arial;">
  <h1 style="margin:0;">📚 AI PDF Assistant</h1>
  <p style="margin:8px 0 0;">Upload knowledge. Ask questions. Learn faster.</p>
</div>
""")

# ---------- Question input ----------
question_box = widgets.Textarea(
    placeholder="Example: What is Artificial Intelligence?",
    description="Question:",
    layout=widgets.Layout(width="100%", height="90px"),
    style={"description_width": "initial"}
)

ask_button = widgets.Button(
    description="Ask AI",
    icon="paper-plane",
    button_style="primary",
    layout=widgets.Layout(width="68%", height="42px")
)

reset_button = widgets.Button(
    description="Clear",
    icon="refresh",
    layout=widgets.Layout(width="28%", height="42px")
)

status = widgets.HTML()
answer_panel = widgets.HTML()

# ---------- Answer generation ----------
def generate_answer(question):
    question_embedding = embedding_model.encode(
        question,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        question_embedding,
        chunk_embeddings
    )[0]

    k = min(3, len(chunks))
    best_matches = scores.topk(k)

    context_parts = [
        chunks[index.item()]
        for index in best_matches.indices
    ]

    context = "\n\n".join(context_parts)

    prompt = f"""
You are an AI assistant answering questions about a PDF.

Use only the document excerpts below to answer.
If the excerpts do not provide enough information,
say that the answer could not be found in the
retrieved excerpts. Do not invent facts.

Document excerpts:
{context}

User question:
{question}

Give a clear, simple answer.
"""

    response = client.models.generate_content(
        model="gemini-flash-lite-latest",
        contents=prompt
    )

    return response.text or "The AI returned an empty answer.", context_parts


def on_ask_clicked(button):
    question = question_box.value.strip()

    if not question:
        status.value = (
            "<p style='color:#b45309;'>"
            "Please enter a question first.</p>"
        )
        answer_panel.value = ""
        return

    if not chunks or len(chunk_embeddings) == 0:
        status.value = (
            "<p style='color:#b91c1c;'>"
            "Please extract your PDF text and create embeddings first.</p>"
        )
        return

    ask_button.disabled = True
    status.value = "<p style='color:#2563eb;'>Thinking... Please wait.</p>"
    answer_panel.value = ""

    try:
        answer, sources = generate_answer(question)

        safe_answer = html.escape(answer)
        safe_question = html.escape(question)

        source_html = ""
        for number, source in enumerate(sources, start=1):
            source_html += f"""
            <details style="margin-top:10px;">
              <summary>Retrieved excerpt {number}</summary>
              <p style="white-space:pre-wrap;">{html.escape(source)}</p>
            </details>
            """

        answer_panel.value = f"""
        <div style="font-family:Arial;padding:20px;
        border:1px solid #cbd5e1;border-radius:14px;
        margin-top:12px;color:#1e293b;">
          <p style="color:#64748b;margin:0;">YOUR QUESTION</p>
          <h3>{safe_question}</h3>
          <hr style="border:0;border-top:1px solid #e2e8f0;">
          <h3>🤖 AI Answer</h3>
          <div style="white-space:pre-wrap;line-height:1.6;">{safe_answer}</div>
          <hr style="border:0;border-top:1px solid #e2e8f0;">
          <h4>📄 Retrieved PDF excerpts</h4>
          {source_html}
        </div>
        """
        status.value = (
            "<p style='color:#15803d;'>Answer generated successfully.</p>"
        )

    except Exception as error:
        status.value = (
            "<p style='color:#b91c1c;'>"
            "Something went wrong. Check the error below.</p>"
        )
        answer_panel.value = (
            "<pre style='white-space:pre-wrap;'>"
            + html.escape(str(error))
            + "</pre>"
        )

    finally:
        ask_button.disabled = False


def on_reset_clicked(button):
    question_box.value = ""
    status.value = ""
    answer_panel.value = ""


ask_button.on_click(on_ask_clicked)
reset_button.on_click(on_reset_clicked)

# ---------- Layout ----------
form = widgets.VBox(
    [
        widgets.HTML("<h3>Ask a question about your PDF</h3>"),
        question_box,
        widgets.HBox(
            [ask_button, reset_button],
            layout=widgets.Layout(justify_content="space-between")
        ),
        status,
        answer_panel
    ],
    layout=widgets.Layout(
        padding="20px",
        border="1px solid #e2e8f0",
        border_radius="14px",
        gap="12px"
    )
)

app = widgets.VBox(
    [header, form],
    layout=widgets.Layout(
        width="100%",
        max_width="800px",
        gap="16px"
    )
)

display(app)
